# Data processing

## I2V model

In [ ]:
!CUDA_VISIBLE_DEVICES="0" python examples/wanvideo/train_wan_t2v.py \
  --task data_process \
  --dataset_path data/example_dataset \
  --output_path ./models \
  --text_encoder_path "models/Wan-AI/Wan2.1-I2V-14B-480P/models_t5_umt5-xxl-enc-bf16.pth" \
  --image_encoder_path "models/Wan-AI/Wan2.1-I2V-14B-480P/models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth" \
  --vae_path "models/Wan-AI/Wan2.1-I2V-14B-480P/Wan2.1_VAE.pth" \
  --tiled \
  --num_frames 29 \
  --height 480 \
  --width 832

The `num_frames` can't be longer than the minimum video length, otherwise it'll fail.

In [ ]:
!CUDA_VISIBLE_DEVICES="0" python examples/wanvideo/train_wan_t2v.py \
  --task data_process \
  --dataset_path data/example_dataset \
  --output_path ./models \
  --text_encoder_path "models/Wan-AI/Wan2.1-I2V-14B-480P/models_t5_umt5-xxl-enc-bf16.pth" \
  --image_encoder_path "models/Wan-AI/Wan2.1-I2V-14B-480P/models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth" \
  --vae_path "models/Wan-AI/Wan2.1-I2V-14B-480P/Wan2.1_VAE.pth" \
  --height 512 \
  --width 512 \
  --tiled \
  --num_frames 29

# LoRA Training for I2V model

In [ ]:
!CUDA_VISIBLE_DEVICES="0" python examples/wanvideo/train_wan_t2v.py \
  --task train \
  --train_architecture lora \
  --dataset_path data/example_dataset \
  --output_path ./models \
  --dit_path "models/Wan-AI/Wan2.1-I2V-14B-480P/diffusion_pytorch_model-00001-of-00007.safetensors,models/Wan-AI/Wan2.1-I2V-14B-480P/diffusion_pytorch_model-00002-of-00007.safetensors,models/Wan-AI/Wan2.1-I2V-14B-480P/diffusion_pytorch_model-00003-of-00007.safetensors,models/Wan-AI/Wan2.1-I2V-14B-480P/diffusion_pytorch_model-00004-of-00007.safetensors,models/Wan-AI/Wan2.1-I2V-14B-480P/diffusion_pytorch_model-00005-of-00007.safetensors,models/Wan-AI/Wan2.1-I2V-14B-480P/diffusion_pytorch_model-00006-of-00007.safetensors,models/Wan-AI/Wan2.1-I2V-14B-480P/diffusion_pytorch_model-00007-of-00007.safetensors" \
  --image_encoder_path "models/Wan-AI/Wan2.1-I2V-14B-480P/models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth" \
  --steps_per_epoch 40 \
  --num_frames 29 \
  --height 512 \
  --width 512 \
  --max_epochs 3 \
  --learning_rate 1e-4 \
  --lora_rank 16 \
  --lora_alpha 16 \
  --lora_target_modules "q,k,v,o,ffn.0,ffn.2" \
  --accumulate_grad_batches 1 \
  --use_gradient_checkpointing

In [ ]:
import torch
from diffsynth import ModelManager, WanVideoPipeline, save_video, VideoData
from modelscope import snapshot_download, dataset_snapshot_download
from PIL import Image

In [ ]:
# Download example image
dataset_snapshot_download(
    dataset_id="DiffSynth-Studio/examples_in_diffsynth",
    local_dir="./",
    allow_file_pattern=f"data/examples/wan/input_image.jpg"
)
image = Image.open("data/examples/wan/input_image.jpg")

# Image-to-Video using LoRA model

In [ ]:
import torch
from diffsynth import ModelManager, WanVideoPipeline, save_video
from torchvision.transforms import v2
from einops import rearrange
from PIL import Image
import imageio
import numpy as np
import matplotlib.pyplot as plt
import cv2

def load_video_as_pil_frames(video_path, num_frames=81):
    """
    Loads a video and returns a list of PIL.Image frames.
    If the video has more frames than `num_frames`, it samples evenly spaced frames.
    """
    reader = imageio.get_reader(video_path)
    total_frames = reader.count_frames()
    print(f"total_frames: {total_frames}")
    assert total_frames >= num_frames, f"Video must have at least {num_frames} frames."

    # Generate evenly spaced frame indices from 0 to total_frames-1
    indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
    # indices = np.linspace(0, num_frames - 1, num=num_frames, dtype=int)

    frames = []
    for i in indices:
        frame = reader.get_data(i)
        frame = Image.fromarray(frame).convert("RGB")
        frames.append(frame)
    reader.close()
    return frames

def replace_rgb_with_black(frames, target_val, tol=1):
    """
    frames: list of PIL.Image in RGB
    target_val: int 0–255, the nominal RGB value you’re matching (e.g. 128)
    tol: tolerance so you match [target_val - tol … target_val + tol]
    Returns: list of PIL.Image with every pixel in that range set to (0,0,0)
    """
    out = []
    low, high = target_val - tol, target_val + tol

    for im in frames:
        arr = np.array(im, dtype=np.uint8)   # shape (H, W, 3)

        # element‐wise mask for each channel in [low..high]
        in_range = (arr >= low) & (arr <= high)     # shape (H, W, 3), bool

        # want pixels where *all* three channels are within range
        mask = np.all(in_range, axis=-1)            # shape (H, W), bool

        # set those pixels to black
        arr[mask] = [0, 0, 0]

        out.append(Image.fromarray(arr, mode="RGB"))
    return out

def plot_distribution(img):
    # 2. Convert to NumPy array
    arr = np.array(img, dtype=np.uint8)   # shape (H, W)
    
    # 3. Plot histogram
    plt.figure(figsize=(8,4))
    plt.hist(arr.ravel(), bins=256, range=(0,255), density=True)
    plt.title('Grayscale Pixel Intensity Distribution')
    plt.xlabel('Pixel value (0=black … 255=white)')
    plt.ylabel('Fraction of pixels')
    plt.xlim(0, 255)
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
def interpolate_video(video_path, num_frames=81, out_path="output.mp4", fps=30):
    """
    Loads a video, evenly interpolates a total of `num_frames` frames from it, and returns
    a list of PIL.Image frames. Also, the interpolated video is saved to `out_path`.

    If the target frame index is non-integer, a simple linear interpolation is performed
    between the two nearest frames.

    Args:
      video_path (str): Path to the input video.
      num_frames (int): Number of frames to sample/interpolate.
      out_path (str): Output file path for saving the new video.
      fps (int): The frame rate for the saved video.

    Returns:
      A list of PIL.Image frames representing the interpolated video.
    """
    reader = imageio.get_reader(video_path)
    total_frames = reader.count_frames()
    print(f"total_frames: {total_frames}")
    
    # Create evenly spaced (possibly fractional) indices from 0 to total_frames-1.
    # These indices may not be integers so we can interpolate if needed.
    indices = np.linspace(0, total_frames - 1, num=num_frames)
    
    frames = []
    for idx in indices:
        lower = int(np.floor(idx))
        upper = int(np.ceil(idx))
        alpha = idx - lower  # fractional part
        
        if lower == upper or alpha == 0:
            # No interpolation needed, directly use the frame.
            frame = reader.get_data(lower)
        else:
            # Retrieve both frames and interpolate.
            frame1 = reader.get_data(lower).astype(np.float32)
            frame2 = reader.get_data(upper).astype(np.float32)
            interp_frame = (1 - alpha) * frame1 + alpha * frame2
            frame = interp_frame.astype(np.uint8)
        
        # Convert the frame to a PIL Image in RGB.
        frame_pil = Image.fromarray(frame).convert("RGB")
        frames.append(frame_pil)
    reader.close()
    
    # Save the new video. Convert PIL Images back to NumPy arrays.
    frames_array = [np.array(frame) for frame in frames]
    imageio.mimsave(out_path, frames_array, fps=fps)
    # print(f"Saved interpolated video to {out_path}")
    return frames

In [ ]:
video = load_video_as_pil_frames("data/mask/kitchen_in_mask.mp4", num_frames=81)
video[2]

In [ ]:
plot_distribution(video[2])

In [ ]:
# 2. Mask out all pixels==128
masked_video = replace_rgb_with_black(video, 128, tol=5)
masked_video[2]

In [ ]:
plot_distribution(masked_video[2])

In [ ]:
interpolate_video("data/avengers.mp4", num_frames=181, out_path="data/avengers_181_frames.mp4", fps=24)

In [ ]:
video = load_video_as_pil_frames("data/example_dataset/train/avengers.mp4", num_frames=181)

In [ ]:
video = load_video_as_pil_frames("data/lighthouse_352_640_12fps.mp4", num_frames=117)

In [ ]:
save_video(video, "data/avengers_416_240_12fps_109.mp4", fps=12, quality=7)

In [ ]:
video = [pad_frame(frame, (832, 832)) for frame in video]

In [ ]:
video_paths = {
    "lioness": "data/lioness_81_frames.mp4",
    "husky": "data/husky_12fps_181frames.mp4",
    "avengers": "data/avengers_24fps_576x288_261.mp4",
    "tom_jerry": "data/tom_jerry_480x304_24fps_277_frames.mp4",
    "road": "data/road_640_352_12fps_185f.mp4",
    "surf": "data/surf_416x240_12fps_77f.mp4",
    "car": "data/car_416_240_12fps_81f.mp4",
    "kitchen": "data/kitchen_416_240_12fps_81f.mp4",
    "city": "data/city_416_240_12fps_81f.mp4",
}
for video_name, video_path in video_paths.items():
    print(video_name, video_path)

In [ ]:
# Standardize videos
num_frames = 81
width=416
height=240

for video_name, video_path in video_paths.items():
    # 1) interpolate to exactly `num_frames` frames
    frames = interpolate_video(video_path, num_frames=81)
    # 2) resize each frame
    frames_resized = [
        frame.resize((width, height), Image.BILINEAR)
        for frame in frames
    ]
    # 3) save with your utility
    save_path = f"data/preprocessed/{video_name}.mp4"
    save_video(frames_resized, save_path, fps=12, quality=5)
    
    print(f"✅  Processed & saved: {save_path}")

In [ ]:
mask_video = VideoData("data/husky_masked_640_368_137f.mp4")
raw_frames = mask_video.raw_data()

In [ ]:
len(raw_frames)

In [ ]:
gray_stack = torch.stack([
    torch.from_numpy(np.array(f.convert("L")))
    for f in raw_frames
], dim=0).unsqueeze(1)  # (T,1,H,W)
gray_stack.size()

In [ ]:
gray_stack[0][0]

In [ ]:
pixel_mask = (gray_stack == 128).float()
count_ones = (pixel_mask == 1).sum().item()
count_zeros = (pixel_mask == 0).sum().item()
print(f"total 1: {count_ones}; total 0: {count_zeros}")

# Video Screenshots

In [ ]:
def resize_image(img, target_w: int = 640, target_h: int = 368):
    orig_w, orig_h = img.size
    # 2. Compute how tall the crop must be in order
    #    to match the target aspect ratio when resized:
    #      aspect = target_w / target_h
    #      crop_h = orig_w / aspect
    aspect = target_w / target_h
    crop_h = int(orig_w / aspect)

    #    If the image is too short, just use full height.
    crop_h = min(crop_h, orig_h)

    #    Center‐crop vertically
    top    = (orig_h - crop_h) // 2
    bottom = top + crop_h
    #    keep full width: left=0, right=orig_w
    img_cropped = img.crop((0, top, orig_w, bottom))

    # 3. Resize to exactly target_w×target_h
    img_resized = img_cropped.resize((target_w, target_h), Image.LANCZOS)
    return img_resized

def print_interpolate_frames(video_path, video_name, central_mask=False, stitch=False, size=None):
    print(video_name)
    reader = imageio.get_reader(video_path)
    total_frames = reader.count_frames()
    # print(total_frames)
    frames = interpolate_video(video_path, num_frames=total_frames)
    selected_frames = [
        frames[2],
        frames[total_frames // 6],
        frames[total_frames // 6 * 2],
        frames[total_frames // 6 * 3],
        frames[total_frames // 6 * 4],
        frames[total_frames - 2]
    ]
    if size:
        selected_frames = [resize_image(img, size[0], size[1]) for img in selected_frames]

    first_frame = selected_frames[0] if isinstance(selected_frames[0], Image.Image) else Image.fromarray(selected_frames[0])
    arr = np.array(first_frame, dtype=np.uint8)          # H×W×3
    height, width = arr.shape[:2]
    
    if central_mask:
        masked = []
        for frame in selected_frames:
            # make sure we’re working on a PIL image
            im = frame if isinstance(frame, Image.Image) else Image.fromarray(frame)
            arr = np.array(im, dtype=np.uint8)          # H×W×3

            h, w = arr.shape[:2]
            # compute central rectangle coords
            x1, y1 = w // 4, h // 4
            x2, y2 = x1 + w // 2, y1 + h // 2

            # paint the central box
            arr[y1:y2, x1:x2] = (128, 128, 128)
            masked.append(Image.fromarray(arr))
        selected_frames = masked

    if stitch:
        # stitch horizontally
        stitched_array = np.hstack(selected_frames)
        stitched_frames = Image.fromarray(stitched_array)
        stitched_frames.show()
    else:
        for i, frame in enumerate(selected_frames):
            # print(i)
            # frame.show()
            frame.save(f"data/frames/{video_name}_{i}.png")
    

In [ ]:
video_dict = {
    # "surf": {
    #     "input": "data/surf_416x240_12fps_77f.mp4",
    #     "mask_fixed": "data/surf_416x240_12fps_77f.mp4",
    #     "inpainting_Ours": "data/output/surf_inpainting_co_denoising_t2v_LoRA_overlap_0_240x416_denoising_0.75_lora_alpha_0.7_steps_50_batch_77f.mp4",
    #     "mask_SAM2": "data/mask/surf_in_mask.mp4",
    #     # "inpainting_Ours_mask_SAM2": "data/output/surf_inpainting_Wan2.1-T2V-14B_mask_True_overlap_0_240x416_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
    #     "outpainting_Ours": "data/output/surf_outpainting_co_denoising_t2v_LoRA_overlap_0_480x832_denoising_0.72_lora_alpha_0.7_steps_50_batch_77f.mp4",
    #     "inpainting_VACE": "data/VACE/surf_VACE_inpainting.mp4",
    #     "outpainting_VACE": "data/VACE/surf_VACE_outpainting.mp4",
    #     "inpainting_WanT2V14B": "data/WanT2V/surf_WanT2V14B_inpainting_masked_steps_50_strength_0.75_416x240_77f.mp4",
    #     "outpainting_WanT2V14B": "data/WanT2V/surf_WanT2V14B_outpainting_masked_steps_50_strength_0.75_416x240_77f.mp4",
    #     "inpainting_VideoXFun": "data/VideoFun/surf_VideoFun_inpainting_masked_steps_50.mp4",
    #     "outpainting_VideoXFun": "data/VideoFun/surf_VideoFun_outpainting.mp4",
    # },
    # "husky": {
    #     "input": "data/husky_12fps_181frames.mp4",
    #     "mask_fixed": "data/husky_12fps_181frames.mp4",
    #     "inpainting_Ours": "data/output/husky_inpaint_co_denoising_t2v_LoRA_v15_368x640_denoising_strength_0.7_lora_alpha_0.7_steps_50.mp4",
    #     "mask_SAM2": "data/mask/husky_in_mask.mp4",
    #     "inpainting_Ours_mask_SAM2": "data/output/husky_inpainting_Wan2.1-T2V-14B_mask_True_overlap_0_368x640_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
    #     "outpainting_Ours": "data/output/husky_outpaint_t2v_LoRA_v15_1024x1024_denoising_strength_0.7_lora_alpha_0.7_steps_30.mp4",
    #     "inpainting_VACE": "data/VACE/husky_VACE_inpainting.mp4",
    #     "outpainting_VACE": "data/VACE/husky_VACE_outpainting.mp4",
    #     "inpainting_WanT2V14B": "data/WanT2V/husky_WanT2V_inpainting.mp4",
    #     "outpainting_WanT2V14B": "data/WanT2V/husky_WanT2V_outpainting.mp4",
    #     "inpainting_VideoXFun": "data/VideoFun/husky_VideoFun_inpainting.mp4",
    #     "outpainting_VideoXFun": "data/VideoFun/husky_VideoFun_outpainting.mp4",
    # },
    # "lioness": {
    #     "input": "data/lioness_864x480_81f.mp4",
    #     "mask_fixed": "data/lioness_864x480_81f.mp4",
    #     "inpainting_Ours": "data/output/lioness_inpaint_t2v_LoRA_v3_480x832_denoising_strength_0.6_lora_alpha_0.7_steps_50.mp4",
    #     "mask_SAM2": "data/mask/lioness_in_mask.mp4",
    #     # "inpainting_Ours_mask_SAM2": "data/output/.mp4",
    #     "outpainting_Ours": "data/output/lioness_outpaint_t2v_LoRA_v8_832x1024_denoising_strength_0.7_lora_alpha_0.7_steps_40.mp4",
    #     "inpainting_VACE": "data/VACE/lioness_VACE_inpainting.mp4",
    #     "outpainting_VACE": "data/VACE/lioness_VACE_outpainting.mp4",
    # },
    # "avengers": {
    #     "input": "data/avengers_416_240_12fps_109f.mp4",
    #     "mask_fixed": "data/avengers_416_240_12fps_109f.mp4",
    #     "inpainting_Ours": "data/output/avengers_inpaint_co_denoising_t2v_LoRA_v23_480x928_denoising_strength_0.65_lora_alpha_0.7_steps_45.mp4",
    #     "mask_SAM2": "data/mask/avengers_in_mask.mp4",
    #     # "inpainting_Ours_mask_SAM2": "data/output/.mp4",
    #     "outpainting_Ours": "data/output/avengers_outpaint_co_denoising_t2v_LoRA_v35_overlap_2_600x1200_denoising_0.7_lora_alpha_0.7_steps_30.mp4",
    #     "inpainting_VACE": "data/VACE/avengers_VACE_inpainting.mp4",
    #     "inpainting_WanT2V14B": "data/WanT2V/avengers_WanT2V_inpainting.mp4",
    #     "outpainting_WanT2V14B": "data/WanT2V/avengers_WanT2V_outpainting.mp4",
    #     "inpainting_VideoXFun": "data/VideoFun/avengers_VideoFun_inpainting.mp4",
    #     "outpainting_VideoXFun": "data/VideoFun/avengers_VideoFun_outpainting.mp4",
    # },
    # "orange": {
    #     "input": "data/orange_368_640_24fps_217f.mp4",
    #     "mask_fixed": "data/orange_368_640_24fps_217f.mp4",
    #     "inpainting_Ours": "data/output/orange_inpaint_co_denoising_t2v_LoRA_overlap_3_640x368_denoising_0.65_lora_alpha_0.6_steps_50_pad_False.mp4",
    #     "mask_SAM2": "data/mask/orange_in_mask.mp4",
    #     # "inpainting_Ours_mask_SAM2": "data/output/.mp4",
    #     "outpainting_Ours": "data/output/orange_outpaint_co_denoising_t2v_LoRA_v35_overlap_3_1280x736_denoising_0.7_lora_alpha_0.7_steps_40_pad_False.mp4",
    #     "inpainting_VACE": "data/VACE/orange_VACE_inpainting.mp4",
    #     "outpainting_VACE": "data/VACE/orange_VACE_outpainting.mp4",
    # },
    # "lighthouse": {
    #     "input": "data/lighthouse_352_640_12fps_117f.mp4",
    #     "mask_fixed": "data/lighthouse_352_640_12fps_117f.mp4",
    #     "inpainting_Ours": "data/output/lighthouse_inpaint_co_denoising_t2v_LoRA_overlap_0_640x352_denoising_0.72_lora_alpha_0.75_steps_100_batch_100f_pad_True_cacheT_4.mp4",
    #     "mask_SAM2": "data/mask/lighthouse_in_mask.mp4",
    #     # "inpainting_Ours_mask_SAM2": "data/output/.mp4",
    #     "outpainting_Ours": "data/output/lighthouse_outpaint_co_denoising_t2v_LoRA_overlap_0_1280x704_denoising_0.78_lora_alpha_0.88_steps_40_batch_100f_pad_False_cacheT_4.mp4",
    #     "inpainting_VACE": "data/VACE/lighthouse_VACE_inpainting.mp4",
    #     "outpainting_VACE": "data/VACE/lighthouse_VACE_outpainting.mp4",
    #     "inpainting_WanT2V14B": "data/WanT2V/lighthouse_WanT2V14B_outpainting_masked_steps_50_strength_0.75_352x640_117f.mp4",
    #     "outpainting_WanT2V14B": "data/WanT2V/lighthouse_WanT2V14B_inpainting_masked_steps_50_strength_0.75_352x640_117f.mp4",
    #     "inpainting_VideoXFun": "data/VideoFun/lighthouse_VideoFun_inpainting.mp4",
    #     "outpainting_VideoXFun": "data/VideoFun/lighthouse_VideoFun_outpainting.mp4",
    # },
    # "road": {
    #     "input": "data/road_640_352_12fps_185f.mp4",
    #     "mask_fixed": "data/road_640_352_12fps_185f.mp4",
    #     "inpainting_Ours": "data/output/road_inpaint_co_denoising_t2v_LoRA_overlap_0_352x640_denoising_0.72_lora_alpha_0.75_steps_100_batch_100f_pad_True_cacheT_4.mp4",
    #     "mask_SAM2": "data/mask/road_in_mask.mp4",
    #     # "inpainting_Ours_mask_SAM2": "data/output/.mp4",
    #     "outpainting_Ours": "data/output/road_outpaint_co_denoising_t2v_LoRA_overlap_6_704x1280_denoising_0.7_lora_alpha_0.7_steps_45_batch_100f_pad_False.mp4",
    #     "inpainting_VACE": "data/VACE/road_VACE_inpainting.mp4",
    #     "outpainting_VACE": "data/VACE/road_VACE_outpainting.mp4",
    # },
    "car": {
        "input": "data/car_416_240_12fps_81f.mp4",
        "mask_fixed": "data/car_416_240_12fps_81f.mp4",
        "inpainting_Ours": "data/output/car_inpainting_co_denoising_t2v_LoRA_overlap_0_240x416_denoising_0.75_lora_alpha_0.7_steps_50_batch_81f.mp4",
        "mask_SAM2": "data/mask/car_in_mask.mp4",
        # "inpainting_Ours_mask_SAM2": "data/output/.mp4",
        "outpainting_Ours": "data/output/car_outpainting_co_denoising_t2v_LoRA_overlap_0_480x832_denoising_0.75_lora_alpha_0.7_steps_50_batch_81f.mp4",
        "inpainting_VACE": "data/VACE/car_VACE_inpainting.mp4",
        "outpainting_VACE": "data/VACE/car_VACE_outpainting.mp4",
        "inpainting_WanT2V14B": "data/WanT2V/car_WanT2V_inpainting.mp4",
        "outpainting_WanT2V14B": "data/WanT2V/car_WanT2V_outpainting.mp4",
        "inpainting_VideoXFun": "data/VideoFun/car_VideoFun_inpainting.mp4",
        "outpainting_VideoXFun": "data/VideoFun/car_VideoFun_outpainting.mp4",
        "ablation_out_Wmasked_09": "data/output/car_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.9_overlap_0_480x832_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
        # "ablation_in_Wmasked_07": "data/output/car_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.7_overlap_0_480x832_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
        "ablation_out_Wmasked_05": "data/output/car_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.5_overlap_0_480x832_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
        # "ablation_in_Wmasked_03": "data/output/car_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.3_overlap_0_480x832_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
        "ablation_out_Wmasked_01": "data/output/car_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.1_overlap_0_480x832_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
        "ablation_in_Wmasked_09": "data/output/car_inpainting_Wan2.1-T2V-14B_mask_True_Wmasked_0.9_overlap_0_240x416_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
        "ablation_in_Wmasked_05": "data/output/car_inpainting_Wan2.1-T2V-14B_mask_True_Wmasked_0.5_overlap_0_240x416_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
        "ablation_in_Wmasked_01": "data/output/car_inpainting_Wan2.1-T2V-14B_mask_True_Wmasked_0.1_overlap_0_240x416_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
    },
    # "kitchen": {
    #     "input": "data/kitchen_416_240_12fps_81f.mp4",
    #     "mask_fixed": "data/kitchen_416_240_12fps_81f.mp4",
    #     "inpainting_Ours": "data/output/kitchen_inpainting_co_denoising_t2v_LoRA_overlap_0_240x416_denoising_0.72_lora_alpha_0.7_steps_50_batch_81f.mp4",
    #     "mask_SAM2": "data/mask/kitchen_in_mask.mp4",
    #     # "inpainting_Ours_mask_SAM2": "data/output/.mp4",
    #     "outpainting_Ours": "data/output/kitchen_outpainting_co_denoising_t2v_LoRA_overlap_0_480x832_denoising_0.64_lora_alpha_0.7_steps_50_batch_81f.mp4",
    #     "inpainting_VACE": "data/VACE/kitchen_VACE_inpainting.mp4",
    #     "outpainting_VACE": "data/VACE/kitchen_VACE_outpainting.mp4",
    #     "inpainting_WanT2V14B": "data/WanT2V/kitchen_WanT2V_inpainting.mp4",
    #     "outpainting_WanT2V14B": "data/WanT2V/kitchen_WanT2V_outpainting.mp4",
    #     "inpainting_VideoXFun": "data/VideoFun/kitchen_VideoFun_inpainting.mp4",
    #     "outpainting_VideoXFun": "data/VideoFun/kitchen_VideoFun_outpainting.mp4",
    # },
    # "city": {
    #     "input": "data/city_416_240_12fps_81f.mp4",
    #     "mask_fixed": "data/city_416_240_12fps_81f.mp4",
    #     "inpainting_Ours": "data/output/city_inpainting_co_denoising_t2v_LoRA_overlap_0_240x416_denoising_0.75_lora_alpha_0.7_steps_50_batch_81f.mp4",
    #     "mask_SAM2": "data/mask/city_in_mask.mp4",
    #     # "inpainting_Ours_mask_SAM2": "data/output/.mp4",
    #     "outpainting_Ours": "data/output/city_outpainting_co_denoising_t2v_LoRA_overlap_0_480x832_denoising_0.64_lora_alpha_0.65_steps_50_batch_81f.mp4",
    #     "inpainting_VACE": "data/VACE/city_VACE_inpainting.mp4",
    #     "outpainting_VACE": "data/VACE/city_VACE_outpainting.mp4",
    #     "inpainting_WanT2V14B": "data/WanT2V/city_WanT2V_inpainting.mp4",
    #     "outpainting_WanT2V14B": "data/WanT2V/city_WanT2V_outpainting.mp4",
    #     "inpainting_VideoXFun": "data/VideoFun/city_VideoFun_inpainting.mp4",
    #     "outpainting_VideoXFun": "data/VideoFun/city_VideoFun_outpainting.mp4",
    # },
    "teapot": {
        "input": "data/teapot_240_416_12fps_85f.mp4",
        "mask_fixed": "data/teapot_240_416_12fps_85f.mp4",
        "inpainting_Ours": "data/output/teapot_inpainting_Wan2.1-T2V-14B_mask_True_overlap_0_416x240_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
        "mask_SAM2": "data/mask/teapot_in_mask.mp4",
        # "inpainting_Ours_mask_SAM2": "data/output/.mp4",
        "outpainting_Ours": "data/output/teapot_outpainting_Wan2.1-T2V-14B_overlap_0_832x480_denoising_0.7_lora_alpha_0.7_steps_50_batch_77f.mp4",
        "inpainting_VACE": "data/VACE/teapot_VACE_inpainting.mp4",
        "outpainting_VACE": "data/VACE/teapot_VACE_outpainting.mp4",
    },
    # "tom_jerry": {
    #     "input": "data/tom_jerry_480x304_24fps_277_frames.mp4",
    #     "mask_fixed": "data/tom_jerry_480x304_24fps_277_frames.mp4",
    #     "inpainting_Ours": "data/output/tom_jerry_inpaint_co_denoising_t2v_LoRA_v35_overlap_2_304x480_denoising_0.7_lora_alpha_0.6_steps_35_pad_False.mp4",
    #     # "mask_SAM2": "data/mask/tom_jerry_in_mask.mp4",
    #     # "inpainting_Ours_mask_SAM2": "data/output/.mp4",
    #     "outpainting_Ours": "data/output/tom_jerry_outpainting_co_denoising_t2v_LoRA_overlap_0_608x960_denoising_0.72_lora_alpha_0.7_steps_40_batch_277f_15fps.mp4",
    # },
    
}

In [ ]:
for video_name, video_info in video_dict.items():
    for video_category, path in video_info.items():
        if video_category == "mask_fixed":
            print_interpolate_frames(path, f"{video_name}/{video_name}_{video_category}", central_mask=True)
        else:
            # print_interpolate_frames(path, f"{video_name}/{video_name}_{video_category}", size=(640, 368))
            print_interpolate_frames(path, f"{video_name}/{video_name}_{video_category}")
        # print_interpolate_frames(video_info["input"], f"{video_name}/{video_name}_input")
        # print_interpolate_frames(video_info["mask_fixed"], f"{video_name}/{video_name}_mask_fixed", central_mask=True)
        # print_interpolate_frames(video_info["inpainting_Ours"], f"{video_name}/{video_name}_inpainting_Ours")
        # print_interpolate_frames(video_info["mask_SAM2"], f"{video_name}/{video_name}_mask_SAM2")
        # print_interpolate_frames(video_info["inpainting_Ours_mask_SAM2"], f"{video_name}/{video_name}_inpainting_Ours_mask_SAM2")
        # print_interpolate_frames(video_info["outpainting_Ours"], f"{video_name}/{video_name}_outpainting_Ours")

In [ ]:
import os, shutil
import cv2
import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import lpips
from cdfvd import fvd

# 2) Initialize the LPIPS model (AlexNet backbone by default)
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
lpips_net = lpips.LPIPS(net='alex').to(device)

def load_video_frames(path):
    """Load all frames from a video as a list of RGB numpy arrays."""
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # convert BGR→RGB
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def reshape_gen_video(ref_frames, gen_frames):
    if (ref_frames[0].shape != gen_frames[0].shape):
        h, w = ref_frames[0].shape[:2]
        gen_frames = [
            cv2.resize(f, (w, h), interpolation=cv2.INTER_LINEAR)
            if f.shape[:2] != (h, w) else f
            for f in gen_frames
        ]
    return ref_frames, gen_frames

def compute_psnr_ssim(ref_frames, gen_frames):
    print("Ref frame shape:", ref_frames[0].shape)
    print("Gen frame shape:", gen_frames[0].shape)
    ref_frames, gen_frames = reshape_gen_video(in_ref_video, gen_video)
    psnr_vals, ssim_vals = [], []
    for ref, gen in zip(ref_frames, gen_frames):
        # PSNR (pixel‐wise)
        psnr_vals.append(
            peak_signal_noise_ratio(ref, gen, data_range=255)
        )

        # SSIM (structure), drop the old multichannel kwarg
        ssim_vals.append(
            structural_similarity(
                ref,
                gen,
                data_range=255,
                channel_axis=-1
            )
        )

    return np.mean(psnr_vals), np.mean(ssim_vals)

def compute_lpips(ref_frames, gen_frames):
    ref_frames, gen_frames = reshape_gen_video(in_ref_video, gen_video)
    scores = []
    for ref, gen in zip(ref_frames, gen_frames):
        # to tensor [1×3×H×W], float, in [–1,1]
        R = (torch.from_numpy(ref).permute(2,0,1).unsqueeze(0).float()/127.5 - 1.0).to(device)
        G = (torch.from_numpy(gen).permute(2,0,1).unsqueeze(0).float()/127.5 - 1.0).to(device)
        with torch.no_grad():
            dist = lpips_net(R, G)
        scores.append(dist.item())
    return float(np.mean(scores))

In [ ]:
# in_masked_weight_dict = {
#     "in_masked_w_0.9": "data/output/car_inpainting_Wan2.1-T2V-14B_mask_True_Wmasked_0.9_overlap_0_240x416_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
#     "in_masked_w_0.5": "data/output/car_inpainting_Wan2.1-T2V-14B_mask_True_Wmasked_0.5_overlap_0_240x416_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
#     "in_masked_w_0.1": "data/output/car_inpainting_Wan2.1-T2V-14B_mask_True_Wmasked_0.1_overlap_0_240x416_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
# }
# out_masked_weight_dict = {
#     "out_masked_w_0.9": "data/output/car_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.9_overlap_0_480x832_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
#     "out_masked_w_0.5": "data/output/car_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.5_overlap_0_480x832_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
#     "out_masked_w_0.1": "data/output/car_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.1_overlap_0_480x832_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
# }
out_masked_weight_dict = {
    "out_masked_w_1.0": "data/output/teapot_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_1.0_overlap_0_832x480_denoising_0.7_lora_alpha_0.7_highorder_True_steps_50.mp4",
    "out_masked_w_0.9": "data/output/teapot_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.9_overlap_0_832x480_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
    "out_masked_w_0.5": "data/output/teapot_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.5_overlap_0_832x480_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
    "out_masked_w_0.1": "data/output/teapot_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.1_overlap_0_832x480_denoising_0.7_lora_alpha_0.7_highorder_True_steps_50.mp4"
}
in_masked_weight_dict = {
    "in_masked_w_1.0": "data/output/teapot_inpainting_Wan2.1-T2V-14B_mask_True_Wmasked_1.0_overlap_0_416x240_denoising_0.7_lora_alpha_0.7_highorder_True_steps_50.mp4",
    "in_masked_w_0.9": "data/output/teapot_inpainting_Wan2.1-T2V-14B_mask_True_Wmasked_0.9_overlap_0_416x240_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
    "in_masked_w_0.5": "data/output/teapot_inpainting_Wan2.1-T2V-14B_mask_True_Wmasked_0.5_overlap_0_416x240_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
    "in_masked_w_0.1": "data/output/teapot_inpainting_Wan2.1-T2V-14B_mask_True_Wmasked_0.1_overlap_0_416x240_denoising_0.7_lora_alpha_0.7_steps_50.mp4",
}
no_high_order_solver_w01_path = "data/output/teapot_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.1_overlap_0_832x480_denoising_0.7_lora_alpha_0.7_highorder_False_steps_50.mp4"
no_high_order_solver_w09_path = "data/output/teapot_outpainting_Wan2.1-T2V-14B_mask_False_Wmasked_0.9_overlap_0_832x480_denoising_0.7_lora_alpha_0.7_highorder_False_steps_50.mp4"

In [ ]:
import evaluate
import numpy as np

# Example usage:
video_name = "teapot"

in_ref_video = load_video_frames(video_dict[video_name]["input"])
out_ref_video = load_video_frames(video_dict[video_name]["input"])

# gen_videos = {
#     "ours": video_dict[video_name]["inpainting_Ours"],
#     "VACE": video_dict[video_name]["inpainting_VACE"],
#     "WanT2V": video_dict[video_name]["inpainting_WanT2V14B"],
#     "VideoXFun": video_dict[video_name]["inpainting_VideoXFun"],
# }
# gen_videos = {
#     "ours": video_dict[video_name]["outpainting_Ours"],
#     "VACE": video_dict[video_name]["outpainting_VACE"],
#     "WanT2V": video_dict[video_name]["outpainting_WanT2V14B"],
#     "VideoXFun": video_dict[video_name]["outpainting_VideoXFun"],
# }
# Initialize evaluator; by default uses I3D features
evaluator = fvd.cdfvd('i3d', ckpt_path=None)
# for key, gen_video_path in gen_videos.items():
for key, gen_video_path in in_masked_weight_dict.items():
    gen_video = load_video_frames(gen_video_path)
    
    psnr, ssim = compute_psnr_ssim(in_ref_video, gen_video)
    print(f"{key} Avg PSNR: {np.mean(psnr):.3f} dB")
    print(f"{key} Avg SSIM: {np.mean(ssim):.3f}")
    score = compute_lpips(in_ref_video, gen_video)
    print(f"{key} Avg LPIPS: {score:.3f}")

    # assume these exist
    # fvd = evaluate.load("fvd") 
    
    # # Supply lists of filepaths or NumPy arrays of videos:
    # # videos = [np.ndarray of shape (T, H, W, C), ...]
    # fvd_score = fvd.compute(predictions=[gen_video], references=[in_ref_video])
    # print(f"{key} Avg FVD: {fvd_score:.2f}")

for key, gen_video_path in out_masked_weight_dict.items():
    gen_video = load_video_frames(gen_video_path)
    
    psnr, ssim = compute_psnr_ssim(out_ref_video, gen_video)
    print(f"{key} Avg PSNR: {np.mean(psnr):.3f} dB")
    print(f"{key} Avg SSIM: {np.mean(ssim):.3f}")
    score = compute_lpips(out_ref_video, gen_video)
    print(f"{key} Avg LPIPS: {score:.3f}")


In [ ]:
gen_video = load_video_frames(no_high_order_solver_w09_path)

psnr, ssim = compute_psnr_ssim(in_ref_video, gen_video)
print(f"Avg PSNR: {np.mean(psnr):.3f} dB")
print(f"Avg SSIM: {np.mean(ssim):.3f}")
score = compute_lpips(in_ref_video, gen_video)
print(f"Avg LPIPS: {score:.3f}")

In [ ]:
from cdfvd import fvd

In [ ]:
import numpy as np
import cv2
from cdfvd import fvd

def load_video_array(path):
    """Load all frames as an (T, H, W, 3) uint8 array."""
    cap, frames = cv2.VideoCapture(path), []
    while True:
        ret, frm = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.stack(frames, 0)

# 1) Paths to your two videos
ref_path = video_dict[video_name]["input"]
gen_path = in_masked_weight_dict["in_masked_w_0.9"]

# 2) Load and wrap in batch dimension
ref_arr = load_video_array(ref_path)[None]  # shape (1, T, H, W, C)
gen_arr = load_video_array(gen_path)[None]
print("Video loaded")

# 3) Save to .npy
np.save('data/ref.npy', ref_arr)
np.save('data/gen.npy', gen_arr)
print("Saved to .npy")

# 4) FVD evaluation
print("fvd.cdfvd")
evaluator = fvd.cdfvd('i3d', ckpt_path=None)
print("load_videos")
real_stats = evaluator.load_videos('ref.npy', data_type='video_numpy')
print("compute_real_stats")
evaluator.compute_real_stats(real_stats)
print("load_videos")
fake_stats = evaluator.load_videos('gen.npy', data_type='video_numpy')
print("compute_fake_stats")
evaluator.compute_fake_stats(fake_stats)
print("compute_fvd_from_stats")
score = evaluator.compute_fvd_from_stats()
print(f"FVD: {score:.2f}")


In [ ]:
evaluator.compute_fake_stats(evaluator.load_videos('./example_videos/'))
score = evaluator.compute_fvd_from_stats()

In [ ]:
# Original
print_interpolate_frames("data/surf_inpainting_co_denoising_t2v_LoRA_overlap_0_240x416_denoising_0.75_lora_alpha_0.7_steps_50_batch_77f.mp4", "surf")

In [ ]:
# Mask: fixed
print_interpolate_frames("data/teapot_240_416_12fps.mp4", True)

In [ ]:
# Inpainting - fixed mask
print_interpolate_frames("data/output/tom_jerry_inpaint_co_denoising_t2v_LoRA_v35_overlap_2_304x480_denoising_0.7_lora_alpha_0.6_steps_35_pad_False.mp4")

In [ ]:
# Mask: SAM2
print_interpolate_frames("data/mask/teapot_in_mask.mp4")

In [ ]:
# Inpainting: SAM2
print_interpolate_frames("data/output/teapot_inpainting_Wan2.1-T2V-14B_mask_True_overlap_0_416x240_denoising_0.7_lora_alpha_0.7_steps_50.mp4")

In [ ]:
# Outpainting
print_interpolate_frames("data/output/teapot_outpainting_Wan2.1-T2V-14B_overlap_0_832x480_denoising_0.7_lora_alpha_0.7_steps_50_batch_77f.mp4")

## Other models

In [ ]:
# VACE: Inpainting
print_interpolate_frames("data/VACE/lighthouse_VACE_inpainting.mp4")

In [ ]:
# VACE: Outpainting
print_interpolate_frames("data/VACE/lighthouse_VACE_outpainting.mp4")

In [ ]:
# Wan2.1 T2V 14B: Inpainting
print_interpolate_frames("data/Wan2.1T2V/lighthouse_Wan2.1T2V14B_inpainting_masked_steps_50.mp4")

In [ ]:
# Wan2.1 T2V 14B: Outpainting
print_interpolate_frames("data/Wan2.1T2V/surf_Wan2.1T2V14B_outpainting_masked_steps_50_strength_0.75_832x480_77f.mp4")

In [ ]:
# VideoFun: Inpainting
print_interpolate_frames("data/VideoFun/lighthouse_VideoFun_inpainting_masked_steps_50.mp4")

In [ ]:
# VideoFun: Outpainting
print_interpolate_frames("data/VideoFun/lighthouse_VideoFun_outpainting_masked_steps_50.mp4")

In [ ]:
frames = interpolate_video("data/mask/car_in_mask.mp4", num_frames=81)

In [ ]:
print(1)
frames[0].show()
print(2)
frames[16].show()
print(3)
frames[32].show()
print(4)
frames[48].show()
print(5)
frames[64].show()
print(6)
frames[80].show()

In [ ]:
frames = interpolate_video("data/car_416_240_12fps_81f.mp4", num_frames=81)